In [ ]:
!pip install -q -U accelerate peft trl bitsandbytes

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:

import os, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer
from trl import SFTTrainer, SFTConfig, DPOTrainer
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
import torch


MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
CACHE_DIR = "./cache"


In [ ]:
from datasets import load_dataset

dataset = load_dataset("5CD-AI/Vietnamese-meta-math-MetaMathQA-40K-gg-translated",split="train")
dataset

README.md:   0%|          | 0.00/118 [00:00<?, ?B/s]

MetaMathQA-40K_vi.json:   0%|          | 0.00/69.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40000 [00:00<?, ? examples/s]

Dataset({
    features: ['response_vi', 'query_vi', 'response_en', 'type', 'query_en'],
    num_rows: 40000
})

In [ ]:
dataset[1]

{'response_vi': 'Có 28 quân domino trong bộ này. Jean và ba người bạn của cô ấy đang chơi nên có tổng cộng 4 người chơi. Để biết mỗi người chơi sẽ nhận được bao nhiêu quân domino, chúng ta chia tổng số quân domino cho số người chơi: 28/4 = 7 Mỗi người chơi sẽ nhận được 7 quân domino. ####7 Đáp án là: 7',
 'query_vi': 'Jean và ba người bạn của cô ấy đang chơi trò chơi domino. Có 28 quân domino trong bộ và Jean muốn mỗi người chơi nhận được số lượng quân domino như nhau. Jean và bạn bè của cô ấy sẽ nhận được bao nhiêu quân domino?',
 'response_en': 'There are 28 dominoes in the set.\nJean and her three friends are playing, so there are a total of 4 players.\nTo find out how many dominoes each player will receive, we divide the total number of dominoes by the number of players: 28 / 4 = 7\nEach player will receive 7 dominoes.\n#### 7\nThe answer is: 7',
 'type': 'GSM_AnsAug',
 'query_en': 'Jean and her three friends are playing a game of dominoes. There are 28 dominoes in the set, and Jea

In [ ]:
model =  AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
)



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
messages = [
    {"role": "system", "content": "You are a math reasoning model."},
    {"role": "user", "content": "..."}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

text

'<|im_start|>system\nYou are a math reasoning model.<|im_end|>\n<|im_start|>user\n...<|im_end|>\n<|im_start|>assistant\n'

In [ ]:
import re
from datasets import load_dataset, Dataset
import random
answer_pattern = re.compile(
    r"(đáp án là:|đáp án là :|câu trả lời là:|câu trả lời là :)\s*(.*)",
    re.IGNORECASE
)

formatted_dataset = []


for item in dataset:

    full_response = item["response_vi"].strip()

    response_lower = full_response.lower()

    match = answer_pattern.search(response_lower)

    if match:

        answer = match.group(2).strip()

        # reasoning = everything before answer pattern
        reasoning = full_response[:match.start()].strip()

        formatted_dataset.append({
            "question": item["query_vi"].strip(),
            "reasoning": reasoning,
            "answer": answer
        })

train_dataset = Dataset.from_list(formatted_dataset)

reasoning_start = "<thinking>"
reasoning_end = "</thinking>"

solution_start = "<answer>"
solution_end = "</answer>"

system_prompt = f"""You are given a problem.

Think about the problem and provide your thought process.
Place it between {reasoning_start} and {reasoning_end}.

Then, provide your final answer between
{solution_start} and {solution_end}.
"""

random.seed(42)

random.shuffle(formatted_dataset)

train_samples = formatted_dataset[:3500]
test_samples = formatted_dataset[3500:4000]

train_dataset = Dataset.from_list(train_samples)
test_dataset = Dataset.from_list(test_samples)


def make_chat_format(x):
    return {
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": x["question"]
            }
        ],
        "reasoning": x["reasoning"],
        "answer": x["answer"]
    }

train_dataset = train_dataset.map(make_chat_format)
test_dataset = test_dataset.map(make_chat_format)

print(train_dataset[0]["messages"])
print("GROUND TRUTH:", train_dataset[0]["answer"])
print("REASONING:", train_dataset[0]["reasoning"])
print(f"\nTotal samples: {len(train_dataset)}")

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[{'role': 'system', 'content': 'You are given a problem.\n\nThink about the problem and provide your thought process.\nPlace it between <thinking> and </thinking>.\n\nThen, provide your final answer between\n<answer> and </answer>.\n'}, {'role': 'user', 'content': 'Jean và ba người bạn của cô ấy đang chơi trò chơi domino với một bộ gồm 28 quân domino. Jean muốn chia đều các quân domino cho tất cả người chơi. Mỗi người chơi sẽ nhận được bao nhiêu quân domino, kể cả Jean?'}]
GROUND TRUTH: 7
REASONING: Tổng cộng có 28 quân domino, Jean và ba người bạn của cô ấy đang chơi trò chơi này. Vậy tổng cộng có 1 + 3 = 4 người chơi. Để phân phối các quân domino một cách đồng đều cho tất cả người chơi, chúng ta cần chia tổng số quân domino cho số người chơi. Như vậy mỗi người chơi sẽ nhận được 28 quân domino/ 4 người chơi = 7 quân domino. Do đó, mỗi người chơi, kể cả Jean, sẽ nhận được 7 quân domino. ####7

Total samples: 3500


In [ ]:
args = dict(
    learning_rate=1e-4,
    num_train_epochs=1,
    max_steps=200,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    bf16=False,
    fp16=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,

    lr_scheduler_type="cosine",
    warmup_steps=5,

    max_length=512,

    gradient_checkpointing=True,

    seed=42,
    report_to="none",
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
def params_stats(model):
  total = sum(p.numel() for p in model.parameters())
  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
  return {"total": total, "trainable":trainable, "ratio_pct": trainable * 100/total}

def apply_chat(example, tokenizer):
    messages = example["messages"]

    # assistant target for SFT
    assistant_response = f"""
<thinking>
{example["reasoning"]}
</thinking>

<answer>
{example["answer"]}
</answer>
""".strip()

    messages = messages + [
        {
            "role": "assistant",
            "content": assistant_response
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

def build_messages(example):
    assistant_response = f"""
<thinking>
{example["reasoning"]}
</thinking>

<answer>
{example["answer"]}
</answer>
""".strip()

    return {
        "messages": example["messages"] + [
            {
                "role": "assistant",
                "content": assistant_response
            }
        ]
    }

train_sft_ds = train_dataset.map(build_messages, remove_columns=train_dataset.column_names)
train_sft_ds[0]

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

{'messages': [{'role': 'system',
   'content': 'You are given a problem.\n\nThink about the problem and provide your thought process.\nPlace it between <thinking> and </thinking>.\n\nThen, provide your final answer between\n<answer> and </answer>.\n'},
  {'role': 'user',
   'content': 'Jean và ba người bạn của cô ấy đang chơi trò chơi domino với một bộ gồm 28 quân domino. Jean muốn chia đều các quân domino cho tất cả người chơi. Mỗi người chơi sẽ nhận được bao nhiêu quân domino, kể cả Jean?'},
  {'role': 'assistant',
   'content': '<thinking>\nTổng cộng có 28 quân domino, Jean và ba người bạn của cô ấy đang chơi trò chơi này. Vậy tổng cộng có 1 + 3 = 4 người chơi. Để phân phối các quân domino một cách đồng đều cho tất cả người chơi, chúng ta cần chia tổng số quân domino cho số người chơi. Như vậy mỗi người chơi sẽ nhận được 28 quân domino/ 4 người chơi = 7 quân domino. Do đó, mỗi người chơi, kể cả Jean, sẽ nhận được 7 quân domino. ####7\n</thinking>\n\n<answer>\n7\n</answer>'}]}

In [ ]:


def sft(output_dir='trl_lora'):
  gc.collect()
  torch.cuda.empty_cache()
  tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
  if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
  tokenizer.padding_side = "right"
  model = AutoModelForCausalLM.from_pretrained(
      MODEL_ID,
      dtype=torch.float16,
      device_map="auto",
      cache_dir=CACHE_DIR
  )
  model.config.use_cache=False

  model = get_peft_model(model, peft_config)
  ps = params_stats(model)
  print(f"trainable={ps['trainable']:,} ({ps['ratio_pct']:.3f}%)")

  config = SFTConfig(
      output_dir=output_dir,
      optim="adamw_torch",
      **args,
      assistant_only_loss=True
  )

  train_sft_ds = train_dataset.map(build_messages, remove_columns=train_dataset.column_names)
  test_sft_ds = test_dataset.map(build_messages, remove_columns=train_dataset.column_names)

  trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=train_sft_ds,
    eval_dataset=test_sft_ds,
    processing_class=tokenizer,
    # formatting_func=lambda ex: apply_chat(ex, tokenizer),
  )

  trainer.train()

  adapter_dir = os.path.join(output_dir, "adapter")
  model.save_pretrained(adapter_dir)
  tokenizer.save_pretrained(adapter_dir)

sft()


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.602135,0.544908
100,0.512916,0.505987
150,0.491825,0.493891
200,0.487717,0.492214


In [ ]:
import re

match_format = re.compile(
    rf"^[\s]{{0,}}"
    rf"{reasoning_start}.+?{reasoning_end}.*?"
    rf"{solution_start}.+?{solution_end}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL
)


def match_format_exactly(completions, **kwargs):
    scores = []

    for completion in completions:
        response = completion

        scores.append(
            1.0 if match_format.search(response) is not None else 0.0
        )

    return scores


def match_format_approximately(completions, **kwargs):
    scores = []

    for completion in completions:
        response = completion

        score = 0.0

        score += 0.125 if response.count(reasoning_start) == 1 else -0.25
        score += 0.125 if response.count(reasoning_end) == 1 else -0.25
        score += 0.125 if response.count(solution_start) == 1 else -0.25
        score += 0.125 if response.count(solution_end) == 1 else -0.25

        scores.append(score)

    return scores


def extract_answer(text):
    match = re.search(
        rf"{solution_start}\s*(.*?)\s*{solution_end}",
        text,
        re.DOTALL
    )

    if match:
        return match.group(1).strip()

    return None


def check_numbers(completions, answer, **kwargs):
    scores = []

    for completion, gt in zip(completions, answer):
        response = completion

        pred = extract_answer(response)

        if pred is None:
            scores.append(-1.0)
            continue

        try:
            pred_num = float(pred.strip().replace(",", ""))
            gt_num = float(gt.strip().replace(",", ""))

            if abs(pred_num - gt_num) < 1e-6:
                scores.append(4.0)
            else:
                scores.append(-1.0)

        except:
            scores.append(-1.0)

    return scores

In [ ]:
train_samples_grpo = formatted_dataset[4000:5000]
test_samples = formatted_dataset[6000:6100]

train_grpo_ds = Dataset.from_list(train_samples)
test_grpo_ds = Dataset.from_list(test_samples)

train_grpo_f = train_grpo_ds.map(make_chat_format)
test_grpo_f = test_grpo_ds.map(make_chat_format)

train_grpo_f[0]

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'question': 'Jean và ba người bạn của cô ấy đang chơi trò chơi domino với một bộ gồm 28 quân domino. Jean muốn chia đều các quân domino cho tất cả người chơi. Mỗi người chơi sẽ nhận được bao nhiêu quân domino, kể cả Jean?',
 'reasoning': 'Tổng cộng có 28 quân domino, Jean và ba người bạn của cô ấy đang chơi trò chơi này. Vậy tổng cộng có 1 + 3 = 4 người chơi. Để phân phối các quân domino một cách đồng đều cho tất cả người chơi, chúng ta cần chia tổng số quân domino cho số người chơi. Như vậy mỗi người chơi sẽ nhận được 28 quân domino/ 4 người chơi = 7 quân domino. Do đó, mỗi người chơi, kể cả Jean, sẽ nhận được 7 quân domino. ####7',
 'answer': '7',
 'messages': [{'role': 'system',
   'content': 'You are given a problem.\n\nThink about the problem and provide your thought process.\nPlace it between <thinking> and </thinking>.\n\nThen, provide your final answer between\n<answer> and </answer>.\n'},
  {'role': 'user',
   'content': 'Jean và ba người bạn của cô ấy đang chơi trò chơi domi

In [ ]:
def make_grpo_format(example, tokenizer):
    prompt = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=True,
    )

    return {
        "prompt": prompt,
        "answer": example["answer"],
    }

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import PeftModel

def train_grpo(
    sft_adapter_dir="trl_lora/adapter",
    output_dir="outputs",
):
    gc.collect()
    torch.cuda.empty_cache()
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map="auto",
        cache_dir=CACHE_DIR,
    )

    base_model.config.use_cache = False

    model = PeftModel.from_pretrained(
        base_model,
        sft_adapter_dir,
        is_trainable=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(sft_adapter_dir)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_grpo_formated = train_grpo_f.map(
        lambda x: make_grpo_format(x, tokenizer),
        remove_columns=train_grpo_ds.column_names,
    )
    test_grpo_formated = test_grpo_f.map(
        lambda x: make_grpo_format(x, tokenizer),
        remove_columns=test_grpo_ds.column_names,
    )
    max_len = max(
        len(
            tokenizer(
                x["prompt"],
                add_special_tokens=False,
            )["input_ids"]
        )
        for x in train_grpo_formated
    )

    max_prompt_length = max_len + 1

    print(f"Max prompt length: {max_prompt_length}")

    training_args = GRPOConfig(
        learning_rate=5e-6,
        weight_decay=5e-4,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        optim="adamw_torch_fused",
        logging_steps=20,


        eval_strategy="steps",
        eval_steps=50,

        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,

        num_generations=4, # grpo hyps
        max_completion_length=256,
        num_train_epochs=1,
        max_steps=200,
        fp16=True,
        bf16=False,
        save_steps=100,
        gradient_checkpointing=True,
        max_grad_norm=0.1,

        output_dir=output_dir,
    )

    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[
            match_format_exactly,
            match_format_approximately,
            check_numbers,
        ],
        args=training_args,
        train_dataset=train_grpo_formated,
        eval_dataset=test_grpo_formated,
    )

    trainer.train()

train_grpo()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Max prompt length: 971


Step,Training Loss,Validation Loss
50,0.096495,0.091732
100,0.083455,0.077000
150,0.080481,0.061306
200,0.060973,0.067601


In [ ]:
def load_grpo_model(
    base_model_id=MODEL_ID,
    adapter_dir="outputs/checkpoint-200",
):
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        cache_dir=CACHE_DIR,
    )

    model = PeftModel.from_pretrained(
        base_model,
        adapter_dir,
    )

    model.eval()

    return model, tokenizer


@torch.inference_mode()
def infer(
    model,
    tokenizer,
    question,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.95,
):
    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return response

In [ ]:
model, tokenizer = load_grpo_model(
    adapter_dir="outputs/checkpoint-200"
)

question = """
Một cửa hàng bán 3 cây bút giá 15 nghìn đồng.
Lan mua 5 cây bút và đưa cho người bán 50 nghìn đồng.
Lan sẽ nhận lại bao nhiêu tiền thừa?
"""
response = infer(
    model,
    tokenizer,
    question
)

print(response)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]